In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.window import Window as Window 


In [ ]:
spark = SparkSession.builder \
    .appName("Retail") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

In [ ]:
retail = spark.read.csv(
    "../data/retail.csv",
    header=True,
    inferSchema=True
)

In [ ]:
retail.show(10)

In [ ]:
retail.printSchema()

In [ ]:
retail.count()

In [ ]:
retail.columns

In [ ]:
retail.select("CustomerID").distinct().count()

In [ ]:
retail.select("Country").distinct().count()

In [ ]:
retail.select("Quantity").agg(F.max("Quantity")).show()


In [ ]:
retail2 = retail.groupBy(
    "CustomerID").agg(
        F.countDistinct("InvoiceNo").alias("NumInvoices")
    )
retail2.select(
    "NumInvoices",
    "CustomerID"
    ).show(10)

In [ ]:
retail3 = retail.groupBy(
    "StockCode").agg(
        F.sum("Quantity").alias("TotalQuantity")
    )
retail3.select(
    "TotalQuantity",
    "StockCode"
    ).orderBy(
        F.desc("TotalQuantity")
    ).show(10)

In [ ]:
retail4 = retail.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")  
)
retail4.select(
    "TotalPrice",
    "Quantity",
    "UnitPrice"
    ).show(10)

In [ ]:
retail5= retail.filter(
    F.col("CustomerID").isNotNull()
)
retail5 = retail5.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
)
retail5 = retail5.groupBy(
    "CustomerID").agg(
    F.sum("TotalPrice").alias("TotalSpent")
)
retail5.orderBy(
        F.desc("TotalSpent")
    ).limit(5).show()

In [ ]:
retail6 = retail.filter(
    F.col("CustomerID").isNotNull()
)
retail6 = retail6.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
)
retail6 = retail6.groupBy(
    "Country").agg(
    F.sum("TotalPrice").alias("TotalSpent"),
    F.countDistinct("InvoiceNo").alias("NumInvoices"))
retail6.orderBy(
        F.desc("TotalSpent")
    ).limit(5).show()

In [ ]:
retail7 = retail.filter(
    F.col("StockCode").isNotNull()).filter(
        F.col("Quantity") > 0)
retail7 = retail7.withColumn(
    "TotalRevenue",
    F.col("Quantity") * F.col("UnitPrice")
)
retail7 = retail7.groupBy(
    "StockCode",
    "Description"
    ).agg(
    F.sum("TotalRevenue").alias("TotalProductRevenue"),
    F.sum("Quantity").alias("TotalProductQuantity")
    )
retail7.orderBy(
        F.desc("TotalProductRevenue")
    ).limit(10).show()

In [ ]:
retail8 = retail.filter(
    F.col("CustomerID").isNotNull()).filter(
        F.col("Country").isNotNull()).filter(
            F.col("Quantity") > 0)
retail8 = retail8.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
)   
retail8 = retail8.groupBy(
    "CustomerID",
    "Country"
    ).agg(
    F.sum("TotalPrice").alias("TotalSpent")
    )
retail8.orderBy(
        F.desc("TotalSpent")
    ).limit(10).show()

In [ ]:
retail9= retail.filter(
    F.col("StockCode").isNotNull()).filter(
        F.col("Country").isNotNull()).filter(
            F.col("Quantity") > 0)
retail9 = retail9.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
)   
ventana = Window.partitionBy("Country").orderBy(F.desc("TotalPrice"))
retail9 = retail9.withColumn(
    "Rank",
    F.rank().over(ventana)
)
retail9.filter(
    F.col("Rank") == 1
).select(
    "Country",
    "StockCode",
    "TotalPrice",
    "Rank"
).show(10)


In [ ]:
retail9 = retail.filter(
    F.col("StockCode").isNotNull()
).filter(
    F.col("Country").isNotNull()
).filter(
    F.col("Quantity") > 0
)

retail9 = retail9.groupBy(
    "Country",
    "StockCode",
    "Description"
).agg(
    F.sum("Quantity").alias("TotalQuantity")
)

ventana = Window.partitionBy("Country").orderBy(
    F.desc("TotalQuantity")
)

retail9 = retail9.withColumn(
    "Rank",
    F.rank().over(ventana)
)

retail9.filter(
    F.col("Rank") == 1
).select(
    "Country",
    "StockCode",
    "Description",
    "TotalQuantity"
).show()

In [ ]:
retail10 = retail.filter(
    F.col("CustomerID").isNotNull()
).filter(
    F.col("Quantity") > 0
)
retail10 = retail10.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
)
retail10 = retail10.groupBy(
    "CustomerID",
    "InvoiceNo"
).agg(
    F.sum("TotalPrice").alias("TotalSpent")
)
ventana = Window.partitionBy("InvoiceNo").orderBy(
    F.desc("TotalSpent")
)
retail10 = retail10.withColumn(
    "Rank",
    F.rank().over(ventana)
)
retail10.filter(
    F.col("Rank") == 1
).orderBy(
    F.desc("TotalSpent")
).limit(5).show()

In [ ]:
retail11 = retail.filter(
    F.col("CustomerID").isNotNull()
).filter(
    F.col("Quantity") > 0
)
retail11 = retail11.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
)   
retail11 = retail11.groupBy(
    "Country",
    "InvoiceNo"
).agg(
    F.sum("TotalPrice").alias("orderTotal")
)
retail11 = retail11.groupBy(
    "Country"
).agg(
    F.avg("orderTotal").alias("AverageSpent")
)

retail11.orderBy(
    F.desc("AverageSpent")
).limit(5).show()

In [ ]:
retail12 = retail.filter(
    F.col("CustomerID").isNotNull()
).filter(
    F.col("Quantity") > 0
)
retail12 = retail12.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
)
retail12 = retail12.groupBy(
    "CustomerID",
    "Country"
).agg(
    F.sum("TotalPrice").alias("TotalSpent")
)
ventana = Window.partitionBy("Country").orderBy(
    F.desc("TotalSpent")
)
retail12 = retail12.withColumn(
    "Rank",
    F.row_number().over(ventana)
)
retail12.filter(
    F.col("Rank") == 1
).orderBy(
    F.desc("TotalSpent")
).show()

In [ ]:
retail.filter(F.col("CustomerID").isNull()).count()

In [ ]:
retail.filter(
    F.col("Description").isNull()
).count()

In [ ]:
retail.filter(
    F.col("UnitPrice").isNull()
).count()

In [ ]:
retail.filter(
    F.col("StockCode").isNull()
).count()

In [ ]:
retail_clean = retail.filter(
    F.col("CustomerID").isNotNull()
).filter(
    F.col("Description").isNotNull()
).filter(
    F.col("UnitPrice").isNotNull()
)
retail_clean.count()

In [ ]:
retail.count()

In [ ]:
print("Original:", retail.count())
print("Limpio:", retail_clean.count())
print("Filas eliminadas:", retail.count() - retail_clean.count())

In [ ]:
retail.printSchema()

In [ ]:
retail_clean = retail.withColumn(
    "CustomerID",
    F.col("CustomerID").cast("string")
)
retail_clean = retail_clean.fillna({
    "CustomerID": "Unknown"
})
retail_clean.show(10)

In [ ]:
retail_clean.filter(
    F.col("CustomerID").isNull()
).count()

In [ ]:
retail_clean =retail.withColumn(
"SaleType",
F.when(
    F.col("UnitPrice") < 0, "Return")
    .when(
    F.col("UnitPrice") > 0, "Sale")
    .otherwise("No Movement")
)
retail_clean.show(10)


In [ ]:
retail_clean.groupBy("SaleType").count().show()

In [ ]:
retailclean2 = retail_clean.withColumn(
    "DataQuality",
    F.when(F.col("UnitPrice") < 0, "Invalid Price")
    .when(F.col("UnitPrice") == 0, "Invalid Quantity")
    .when(F.col("UnitPrice") > 10000, "Suspicious Price")
    .otherwise(" OK")
)
retailclean2.groupBy("DataQuality").count().show()

In [ ]:
retail13 = retail.filter(
    F.col("CustomerID").isNotNull()
).filter(
    F.col("Quantity") > 0
).filter(
    F.col("UnitPrice") > 0
)
retail13 = retail13.withColumn(
    "UnitPrice", F.col("UnitPrice").cast("double")
)
retail13 = retail13.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
)
retail13 = retail13.withColumn(
    "year",
    F.year("InvoiceDate")
).withColumn(
    "month",
    F.month("InvoiceDate")
)
retail13 = retail13.groupBy(
    "year",
    "month"
).agg(
    F.sum("TotalPrice").alias("TotalRevenue")
)
retail13.orderBy(
    F.desc("TotalRevenue")
).show()

In [ ]:
retail14 = retail.filter(
    F.col("CustomerID").isNotNull()
).filter(
    F.col("Quantity") > 0
).filter(
    F.col("UnitPrice") > 0
).filter(F.year("InvoiceDate") == 2011)
retail14 = retail14.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
)
retail14 = retail14.withColumn(
    "month",
    F.month("InvoiceDate")
)
retail14 = retail14.groupBy(
    "month",
    "InvoiceNo"
).agg(
    F.sum("TotalPrice").alias("OrderTotal"),
)
retail14 = retail14.groupBy(
    "month"
).agg(
    F.sum("OrderTotal").alias("TotalRevenue"),
    F.countDistinct("InvoiceNo").alias("NumInvoices"),
    F.avg("OrderTotal").alias("AvgInvoiceValue")
)
retail14.orderBy(
    F.asc("month")
).show()

In [ ]:
retail14 = retail.filter(
    F.col("CustomerID").isNotNull()
).filter(
    F.col("Quantity") > 0
).filter(
    F.col("UnitPrice") > 0
)
clientes_2010 = retail14.filter(
    F.year("InvoiceDate") == 2010)
clientes_2011 = retail14.filter(
    F.year("InvoiceDate") == 2011)
clientes_inactivos = clientes_2010.select("CustomerID").distinct().subtract(
    clientes_2011.select("CustomerID").distinct()
)
clientes_inactivos.show(10)

In [ ]:
# 1. Detectar duplicados
duplicados = retail.groupBy(
    "InvoiceNo",
    "StockCode",
    "InvoiceDate"
).count().filter(
    F.col("count") > 1
)

duplicados.show()

In [ ]:
retail_dedup = retail.dropDuplicates([
    "InvoiceNo",
    "StockCode",
    "InvoiceDate"
])

In [ ]:
original = retail.count()
despues = retail_dedup.count()
eliminados = original - despues

print("Original:", original)
print("Después:", despues)
print("Duplicados eliminados:", eliminados)     

In [ ]:
retail17 = retail.withColumn(
"SaleCategory",
F.when(
    F.col("Quantity") < 0, "Return")
    .when(F.col("Quantity") == 0, "No Movement")
    .when(F.col("Quantity") > 10, "Big Sale")
    .otherwise("Small Sale")
)
retail17.groupBy("SaleCategory").count().show()

In [ ]:
retail18 = retail.withColumn(
    "DataQuality",
    F.when(
        (F.col("Quantity") <= 0) | (F.col("UnitPrice") <= 0),
        "Review"
    ).otherwise("OK")
)
retail18.groupBy("DataQuality").count().show()

In [ ]:
retail_valid = retail18.filter(
    F.col("DataQuality") == "OK"
)
retail_errors = retail18.filter(
    F.col("DataQuality") == "Review"
)
print("Filas válidas:", retail_valid.count())
print("Filas con errores:", retail_errors.count())

In [ ]:
retail20 = retail.withColumn(
    "CustomerID",
    F.col("CustomerID").cast("string")
)
retail20 = retail20.fillna({
    "CustomerID": "Unknown"
})
retail20.filter(
    F.col("CustomerID") == "Unknown"
).count()

In [ ]:
retail21 = retail.filter(
    F.col("Quantity") > 0
).filter(
    F.year("InvoiceDate") == 2011
)
retail21 = retail21.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
).withColumn(
    "month",
    F.month("InvoiceDate")
)
retail21 = retail21.groupBy(
    "month",
    "InvoiceNo"
).agg(
    F.sum("TotalPrice").alias("OrderTotal")
)

retail21 = retail21.groupBy(
    "month",
).agg(
    F.sum("OrderTotal").alias("TotalRevenue"),
    F.countDistinct("InvoiceNo").alias("NumInvoices"),
    F.avg("OrderTotal").alias("AvgInvoiceValue")
)
retail21.orderBy(
    F.asc("month")
).show()

In [ ]:
ventana = Window.orderBy("Month")

retail22 = retail21.withColumn(
    "PreviousMonthRevenue",
    F.lag("TotalRevenue").over(ventana)
)

retail22 = retail22.withColumn(
    "Growth",
    F.col("TotalRevenue") - F.col("PreviousMonthRevenue")
)

retail22.orderBy("Month").show()


In [ ]:
retail23 = retail.filter(
    F.col("Quantity") > 0
).filter(
    F.year("InvoiceDate") == 2011
)
retail23 = retail23.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
).withColumn(
    "month",
    F.month("InvoiceDate")
)
retail23 = retail23.groupBy(
    "month",
    "Country").agg(
    F.sum("TotalPrice").alias("TotalRevenue")
)
ventana = Window.partitionBy("Country").orderBy("month")
retail23 = retail23.withColumn(
    "PreviousMonthRevenue",
    F.lag("TotalRevenue").over(ventana)
)
retail23 = retail23.withColumn(
    "Growth",
    F.col("TotalRevenue") - F.col("PreviousMonthRevenue")
)
retail23.filter(
    F.col("Growth") > 0
).orderBy(
    "Country",
    "month"
).show(10)

In [ ]:
retail24 = retail.filter(
    F.col("Quantity") > 0
).filter(
    F.year("InvoiceDate") == 2011
)
retail24 = retail24.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
).withColumn(
    "month",
    F.month("InvoiceDate")
)
retail24 = retail24.groupBy(
    "month",
    "Country").agg(
    F.sum("TotalPrice").alias("TotalRevenue")
)
ventana = Window.partitionBy("Country").orderBy("month")
retail24 = retail24.withColumn(
    "PreviousMonthRevenue",
    F.lag("TotalRevenue").over(ventana)
) 
retail24 = retail24.withColumn(
    "GrowthPercent",
    (
        (F.col("TotalRevenue") - F.col("PreviousMonthRevenue"))
        / F.col("PreviousMonthRevenue")
    ) * 100
)


retail24 = retail24.filter(
    F.col("GrowthPercent") > 20
).filter(
    F.col("PreviousMonthRevenue").isNotNull()
)
retail24.orderBy(
    F.col("GrowthPercent").desc()
).show()

In [ ]:
retail25 = retail.filter(
    F.col("Quantity") > 0
).filter(
    F.year("InvoiceDate") == 2011
)
retail25 = retail25.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
).withColumn(
    "month",
    F.month("InvoiceDate")
)
retail25 = retail25.groupBy(
    "month"
).agg(
    F.sum("TotalPrice").alias("TotalRevenue")
)
ventana = Window.orderBy("month").rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)
retail25 = retail25.withColumn(
    "CumulativeRevenue",
    F.sum("TotalRevenue").over(ventana)
)
retail25.select(
    "month",
    "TotalRevenue",
    "CumulativeRevenue"
).show()

In [ ]:
retail26 = retail.filter(
    F.col("Quantity") > 0
).filter(
    F.year("InvoiceDate") == 2011
)
retail26 = retail26.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
).withColumn(
    "month",
    F.month("InvoiceDate")
)
retail26 = retail26.groupBy(
    "month",
    "Country"
).agg(
    F.sum("TotalPrice").alias("TotalRevenue")
)
ventana = Window.partitionBy(
    "Country"
    ).orderBy(
    "month"
    ).rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
retail26 = retail26.withColumn(
    "CumulativeRevenue",
    F.sum("TotalRevenue").over(ventana)
)
retail26.show()

In [ ]:
retail27 = retail.filter(
    F.col("Quantity") > 0
).filter(
    F.col("CustomerID").isNotNull()
).withColumn(
    "month",
    F.month("InvoiceDate")
)
retail27 = retail27.withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
)
retail27 = retail27.groupBy(
    "Country"
).pivot(
    "month"
).agg(
    F.sum("TotalPrice")
)
retail27.fillna(0).show()

In [ ]:
retail28 = retail.filter(
    F.col("Quantity") > 0
).filter(
    F.col("CustomerID").isNotNull()
).withColumn(
    "month",
    F.month("InvoiceDate")
).withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col("UnitPrice")
)
retail28 = retail28.groupBy(
    "month",
    "Country"
).agg(
    F.sum("TotalPrice").alias("TotalRevenue")
)
ventana = Window.partitionBy("Country").orderBy(F.desc("TotalRevenue"))
retail28 = retail28.withColumn(
    "rank",
    F.row_number().over(ventana)
)
retail28.filter(
    F.col("rank") == 1
).show()

In [ ]:
retail29 = retail.filter(
    F.col("Quantity") > 0
).filter(
    F.col("CustomerID").isNotNull()
).filter(
    F.year("InvoiceDate") == 2011
).withColumn(
    "TotalPrice",
    F.col("Quantity") * F.col ("UnitPrice")
)
retail29 = retail29.groupBy(
    "CustomerID"
).agg(
    F.sum("TotalPrice").alias("Spent")
)
retail29.orderBy(
    F.desc("Spent")
).show(5)


In [ ]:
retail30 = retail.filter(
    F.col("Quantity") > 0
).filter(
    F.col("Country").isNotNull()
).filter(
    F.year("InvoiceDate") == 2011
).filter(
    F.col("StockCode").isNotNull()
)
retail30 = retail30.groupBy(
    "Country",
    "Description",
    "StockCode"
).agg(
    F.sum("Quantity").alias("TotalQuantity")
)
ventana = Window.partitionBy("Country").orderBy(
    F.desc("TotalQuantity")
)
retail30 = retail30.withColumn(
    "rank",
    F.row_number().over(ventana)
)
retail30.filter(
    F.col("rank") == 1
).orderBy(
    F.desc("TotalQuantity")
).show()